# Atualização do Ambiente PIP

In [1]:
%pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Passo 1 - Instalação das Bibliotecas

In [2]:
# Instala as bibliotecas necessárias
%pip install psycopg2-binary
#%pip install psycopg2 - Para evitar falhas de compilação C no Windows/Docker ao conectar no PostgreSQLsubstitui pelo de cima
%pip install pandas
%pip install requests
%pip install minio
%pip install python-dotenv
%pip install sqlalchemy --quiet
#-biquietnary pandas minio pytthon-dotenv sqlalchemy --


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Passo 2 - Importes e Configuração do Ambiente

In [2]:
import os
import json
import requests
import psycopg2
from datetime import datetime
from io import BytesIO
from minio import Minio
from dotenv import load_dotenv

load_dotenv()

# --- Configurações MinIO (Data Lake) ---
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'localhost:9000').replace('http://', '').replace('https://', '')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BUCKET           = 'moedas'
CAMADA           = 'bronze'

# --- Configurações PostgreSQL (Staging Relacional) ---
PG_CONFIG = {
    "dbname": os.getenv("PG_DB", "postgres"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD", "postgres"),
    "host": os.getenv("PG_HOST", "localhost"),
    "port": os.getenv("PG_PORT", "5434")
}

# --- Parâmetros de Busca (AwesomeAPI) ---
MOEDAS_MONITORADAS = "USD-BRL,EUR-BRL,BTC-BRL"
URL_API = f"https://economia.awesomeapi.com.br/last/{MOEDAS_MONITORADAS}"

print("Parâmetros e configurações da Camada Bronze carregados com sucesso!")
print(f"MinIO Endpoint : {MINIO_ENDPOINT}")
print(f"Bucket Target  : {BUCKET}")
print(f"Endpoint API   : {URL_API}")

Parâmetros e configurações da Camada Bronze carregados com sucesso!
MinIO Endpoint : localhost:9000
Bucket Target  : moedas
Endpoint API   : https://economia.awesomeapi.com.br/last/USD-BRL,EUR-BRL,BTC-BRL


# Passo 3 - Conexão e Funções Auxiliares
Inicializa o cliente do MinIO e testa a conectividade com o banco de dados PostgreSQL

In [3]:
def get_minio_client():
    return Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=False
    )

def testar_conexoes():
    # 1. Teste MinIO
    client = get_minio_client()
    if not client.bucket_exists(BUCKET):
        client.make_bucket(BUCKET)
        print(f"Bucket [{BUCKET}] criado no MinIO!")
    else:
        print(f"MinIO OK — Bucket [{BUCKET}] localizado.")

    # 2. Teste Postgres
    conn = psycopg2.connect(**PG_CONFIG)
    conn.close()
    print("PostgreSQL OK — Conexão estabelecida com sucesso.")

testar_conexoes()

MinIO OK — Bucket [moedas] localizado.
PostgreSQL OK — Conexão estabelecida com sucesso.


# Passo 4 - DDL: Criação da Tabela Bronze no PostgreSQL
Cria a estrutura da tabela de staging na Bronze no PostgreSQL (caso ainda não exista).

In [4]:
def criar_tabela_bronze():
    query_ddl = """
    CREATE TABLE IF NOT EXISTS public.extracao_moedas_bronze (
        id_extracao VARCHAR(50),
        codigo_moeda VARCHAR(10),
        nome_moeda VARCHAR(100),
        valor_compra NUMERIC(15, 4),
        valor_venda NUMERIC(15, 4),
        alta NUMERIC(15, 4),
        baixa NUMERIC(15, 4),
        variacao NUMERIC(15, 4),
        pct_mudanca NUMERIC(10, 4),
        data_cotacao TIMESTAMP,
        data_atualizacao TIMESTAMP,
        PRIMARY KEY (id_extracao, codigo_moeda)
    );
    """
    conn = psycopg2.connect(**PG_CONFIG)
    cursor = conn.cursor()
    cursor.execute(query_ddl)
    conn.commit()
    cursor.close()
    conn.close()
    print("Tabela 'public.extracao_moedas_bronze' verificada/criada com sucesso no PostgreSQL!")

criar_tabela_bronze()

Tabela 'public.extracao_moedas_bronze' verificada/criada com sucesso no PostgreSQL!


# Passo 5 - Ingestão Principal (AwesomeAPI ➔ MinIO + PostgreSQL)
Executa a requisição da API, faz o dump do JSON bruto no MinIO e grava a cópia no banco.

In [5]:
def rodar_pipeline_bronze():
    id_extracao = datetime.now().strftime('%Y%m%d%H%M%S')
    data_atualizacao = datetime.now()
    
    print(f" [ID Extração: {id_extracao}] Solicitando cotações na AwesomeAPI...")
    
    try:
        response = requests.get(URL_API, timeout=15)
        
        if response.status_code == 200:
            dados_brutos = response.json()
            
            # --- 1. Gravação Relacional no PostgreSQL ---
            conn = psycopg2.connect(**PG_CONFIG)
            cursor = conn.cursor()
            
            query_insert = """
                INSERT INTO public.extracao_moedas_bronze 
                (id_extracao, codigo_moeda, nome_moeda, valor_compra, valor_venda, alta, baixa, variacao, pct_mudanca, data_cotacao, data_atualizacao)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (id_extracao, codigo_moeda) DO NOTHING;
            """
            
            for chave, item in dados_brutos.items():
                codigo = item.get('code')
                nome = item.get('name')
                bid = float(item.get('bid', 0.0))
                ask = float(item.get('ask', 0.0))
                high = float(item.get('high', 0.0))
                low = float(item.get('low', 0.0))
                var_bid = float(item.get('varBid', 0.0))
                pct_change = float(item.get('pctChange', 0.0))
                data_cotacao = item.get('create_date')
                
                cursor.execute(query_insert, (
                    id_extracao, codigo, nome, bid, ask, high, low, var_bid, pct_change, data_cotacao, data_atualizacao
                ))
                
                print(f"   ->  {codigo} ({nome}): R$ {bid:,.2f} | Variação: {pct_change}%")
                
            conn.commit()
            cursor.close()
            conn.close()
            
            # --- 2. Gravação do JSON Bruto no MinIO (Data Lake) ---
            minio_client = get_minio_client()
            
            payload_minio = {
                "id_extracao": id_extracao,
                "data_atualizacao": data_atualizacao.isoformat(),
                "origem": "AwesomeAPI",
                "dados_brutos": dados_brutos
            }
            
            conteudo_bytes = json.dumps(payload_minio, ensure_ascii=False).encode('utf-8')
            caminho_minio = f"{CAMADA}/ano={data_atualizacao.year}/mes={data_atualizacao.month:02d}/dia={data_atualizacao.day:02d}/cotacao_{id_extracao}.json"
            
            minio_client.put_object(
                BUCKET,
                caminho_minio,
                BytesIO(conteudo_bytes),
                length=len(conteudo_bytes),
                content_type='application/json'
            )
            
            print(f"\n   -> JSON Bruto armazenado no MinIO: s3://{BUCKET}/{caminho_minio}")
            print(" PIPELINE BRONZE CONCLUÍDO COM SUCESSO!")
            
        else:
            print(f"Erro HTTP na AwesomeAPI. Status: {response.status_code}")
            
    except Exception as e:
        print(f"Falha durante a execução do pipeline Bronze: {e}")

rodar_pipeline_bronze()

 [ID Extração: 20260829095359] Solicitando cotações na AwesomeAPI...
   ->  USD (Dólar Americano/Real Brasileiro): R$ 5.18 | Variação: 0.443621%
   ->  EUR (Euro/Real Brasileiro): R$ 6.01 | Variação: -0.161252%
   ->  BTC (Bitcoin/Real Brasileiro): R$ 404,830.00 | Variação: -1.568%

   -> JSON Bruto armazenado no MinIO: s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829095359.json
 PIPELINE BRONZE CONCLUÍDO COM SUCESSO!


# Passo 6 - Validação e Auditoria da Camada Bronze
Inspeciona os arquivos salvos no MinIO e faz uma contagem rápida no banco de dados para garantir que os dados foram persistidos.

In [7]:
import pandas as pd

def auditoria_bronze():
    print("── AUDITORIA DA CAMADA BRONZE ──\n")
    
    # 1. Contagem de registros no PostgreSQL
    conn = psycopg2.connect(**PG_CONFIG)
    df_count = pd.read_sql("SELECT codigo_moeda, COUNT(*) as qtd_extracoes, MAX(data_atualizacao) as ultima_atualizacao FROM public.extracao_moedas_bronze GROUP BY codigo_moeda;", conn)
    conn.close()
    
    print("Registros no PostgreSQL Bronze:")
    print(df_count.to_string(index=False))
    
    # 2. Arquivos JSON gravados no MinIO
    client = get_minio_client()
    objetos = list(client.list_objects(BUCKET, prefix='bronze/', recursive=True))
    
    print(f"\nArquivos JSON acumulados no MinIO ({len(objetos)} arquivo(s)):")
    for obj in sorted(objetos, key=lambda x: x.object_name)[-5:]: # Mostra os últimos 5 arquivos
        print(f"   s3://{BUCKET}/{obj.object_name:<70} | {obj.size / 1024:>5.1f} KB")

auditoria_bronze()

── AUDITORIA DA CAMADA BRONZE ──

Registros no PostgreSQL Bronze:
codigo_moeda  qtd_extracoes         ultima_atualizacao
         BTC              1 2026-08-29 09:53:59.946410
         USD              1 2026-08-29 09:53:59.946410
         EUR              1 2026-08-29 09:53:59.946410

Arquivos JSON acumulados no MinIO (23 arquivo(s)):
   s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829020004.json              |   0.9 KB
   s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829030003.json              |   0.9 KB
   s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829095359.json              |   0.9 KB
   s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829111834.json              |   0.9 KB
   s3://moedas/bronze/ano=2026/mes=08/dia=29/cotacao_20260829120001.json              |   0.9 KB


C:\Users\Admin\AppData\Local\Temp\ipykernel_16768\3640785402.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql("SELECT codigo_moeda, COUNT(*) as qtd_extracoes, MAX(data_atualizacao) as ultima_atualizacao FROM public.extracao_moedas_bronze GROUP BY codigo_moeda;", conn)
